# MetadataSender And MetadataSenderOptions

`pyneat.MetadataSender` sends UTF-8 JSON over UDP. That is the whole feature.
It is how a Neat application tells a viewer, recorder, or downstream service *what it saw* in a frame,
as opposed to `video_sender`, which tells them *what the frame looked like*.

The two travel on separate wires and meet again at the receiver:

```text
video_sender      ->  H.264/RTP  ->  UDP 9000 + channel  ->  Insight draws the picture
MetadataSender    ->  JSON       ->  UDP 9100 + channel  ->  Insight draws boxes on top of it
```

This notebook covers the option surface, the wire contract, the JSON payload shapes that the Insight
viewer knows how to draw, and how to verify metadata is really arriving.

## The One Thing That Surprises People

**`MetadataSender` is not a graph node.** You do not `graph.add(...)` it, you do not connect it, and it
never appears in `describe()`.

It is a plain object your application code holds and calls directly:

```python
sender = pyneat.MetadataSender(options)
sender.send_metadata("object-detection", data_json, timestamp_ms, frame_id)
```

That design is deliberate. Metadata is produced by *your* logic, after you have looked at model output
and decided what it means. There is no pipeline stage that can do that for you, so there is no node.
`video_sender` lives in the graph because pixels flow through the graph. Metadata does not.

The practical consequence: a metadata send happens in your Python loop, in whatever thread you call it
from, and it is fire-and-forget. UDP does not tell you whether anyone was listening.

## Imports

Run this notebook from the DevKit pyneat environment.

In [1]:
import json
import socket
import ssl
import time
import urllib.request

import pyneat

## Notebook Settings

| Setting | Meaning |
| --- | --- |
| `INSIGHT_HOST` | IP of the machine running Insight. Must be reachable from the DevKit. |
| `CHANNEL` | Insight viewer channel, `0`-`79`. **Must match the video channel** for the overlay to land on the right tile. |
| `METADATA_PORT_BASE` | Base UDP port for metadata. Insight's default is `9100`. |
| `FRAME_WIDTH/HEIGHT` | The pixel space your coordinates are in. The viewer scales from this to the tile size. |

In [2]:
INSIGHT_HOST = "192.168.131.12"  # Machine running Insight. Replace with your host IP.
CHANNEL = 0  # Must equal the video_sender channel for the same stream.
METADATA_PORT_BASE = 9100  # Insight metadata UDP base port.

FRAME_WIDTH = 1280  # Coordinate space of the bboxes/keypoints you will send.
FRAME_HEIGHT = 720

INSIGHT_API = f"https://{INSIGHT_HOST}:9900"

print("metadata will be sent to:", f"{INSIGHT_HOST}:{METADATA_PORT_BASE + CHANNEL}")

metadata will be sent to: 192.168.131.12:9100


## Public Fields

The surface is small. Two classes, and one of them has three fields.

In [4]:
def public_names(obj):
    out = [name for name in dir(obj) if not name.startswith("_")]
    return out, len(out)


opt = pyneat.MetadataSenderOptions()

print("MetadataSenderOptions:")
print(public_names(opt))
print()
print("MetadataSender:")
print(public_names(pyneat.MetadataSender))

MetadataSenderOptions:
(['channel', 'host', 'metadata_port_base'], 3)

MetadataSender:
(['host', 'metadata_port', 'ok', 'send_metadata', 'send_raw_json'], 5)


## Complete MetadataSenderOptions Field Map

| Field | Type / default | Use |
| --- | --- | --- |
| `host` | `str`, `"127.0.0.1"` | Destination IP. The machine running Insight, or any UDP JSON receiver. |
| `channel` | `int`, `0` | Logical stream index. Added to `metadata_port_base` to get the real port. |
| `metadata_port_base` | `int`, `9100` | Base UDP port. Insight listens on `9100-9179`. |

That is the entire options object. The port rule is `metadata_port = metadata_port_base + channel`,
exactly parallel to `video_sender`.

## MetadataSender Methods

| Method | Returns | Use |
| --- | --- | --- |
| `MetadataSender(options)` | sender | Opens the UDP socket. Raises on a bad host or a socket failure. |
| `ok()` | `bool` | `True` when the socket is live. |
| `host()` | `str` | The resolved destination host. |
| `metadata_port()` | `int` | The computed destination port. Read the port from here, never by hand. |
| `send_metadata(type, data_json, timestamp_ms, frame_id)` | `bool` | Wraps your data in the standard envelope and sends it. **Use this.** |
| `send_raw_json(payload)` | `bool` | Sends your string verbatim. Only when you already built the whole envelope. |

Construction is where failures surface. An unreachable host name or an unusable socket raises at
construction, not at send time, so build the sender once at startup and let it fail loudly there.

In [6]:
def make_metadata_options(channel: int, host: str = INSIGHT_HOST):
    opt = pyneat.MetadataSenderOptions()
    opt.host = host  # Where Insight is listening.
    opt.channel = channel  # Same channel number as the video stream.
    opt.metadata_port_base = METADATA_PORT_BASE  # Insight metadata base port.
    return opt


sender = pyneat.MetadataSender(make_metadata_options(CHANNEL))

print("ok :", sender.ok())
print("host :", sender.host())
print("port :", sender.metadata_port())
print("expected:", METADATA_PORT_BASE + CHANNEL)

ok : True
host : 192.168.131.12
port : 9100
expected: 9100


## Channel And Port Math

Same rule as video, different base. The reason both exist is that a single channel number then
describes an entire stream: its pixels and its meaning.

| Channel | Video UDP | Metadata UDP | Viewer tile |
| --- | --- | --- | --- |
| `0` | `9000` | `9100` | `src=0` |
| `1` | `9001` | `9101` | `src=1` |
| `7` | `9007` | `9107` | `src=7` |

If the numbers drift apart, the failure is silent and confusing: the video plays, the metadata arrives,
and the boxes are drawn over the *wrong* video. Derive both from one variable.

In [7]:
for ch in [0, 1, 2, 7]:
    meta = pyneat.MetadataSender(make_metadata_options(ch))
    print(f"channel {ch}: video 9000+{ch} = {9000 + ch}   metadata {meta.metadata_port()}   tile src={ch}")

channel 0: video 9000+0 = 9000   metadata 9100   tile src=0
channel 1: video 9000+1 = 9001   metadata 9101   tile src=1
channel 2: video 9000+2 = 9002   metadata 9102   tile src=2
channel 7: video 9000+7 = 9007   metadata 9107   tile src=7


## The Wire Contract

Every payload on the wire is a single UTF-8 JSON object in one UDP datagram. Two top-level fields are
required by every receiver, including Insight:

| Field | Required | Meaning |
| --- | --- | --- |
| `type` | yes | What kind of metadata this is. Drives which overlay the viewer draws. |
| `data` | yes | The payload. Its shape depends on `type`. |
| `timestamp` | no | When the observation was made. |
| `frame_id` | no | Which frame it came from. |

`send_metadata(...)` builds that envelope for you. You pass the *inside* of `data`, and it wraps it.

In [8]:
detections = {
    "objects": [
        {"id": "obj_1", "label": "car", "confidence": 0.92, "bbox": [120, 80, 96, 64]},
        {"id": "obj_2", "label": "person", "confidence": 0.88, "bbox": [420, 300, 60, 120]},
    ]
}

ok = sender.send_metadata(
    "object-detection",          # -> becomes the top-level "type"
    json.dumps(detections),      # -> becomes the top-level "data"
    int(time.time() * 1000),     # -> becomes the top-level "timestamp"
    "frame-7",                   # -> becomes the top-level "frame_id"
)

print("sent:", ok)
print()
print("the datagram that went on the wire:")
print(json.dumps({
    "type": "object-detection",
    "timestamp": int(time.time() * 1000),
    "frame_id": "frame-7",
    "data": detections,
}, indent=2))

sent: True

the datagram that went on the wire:
{
  "type": "object-detection",
  "timestamp": 1784186043777,
  "frame_id": "frame-7",
  "data": {
    "objects": [
      {
        "id": "obj_1",
        "label": "car",
        "confidence": 0.92,
        "bbox": [
          120,
          80,
          96,
          64
        ]
      },
      {
        "id": "obj_2",
        "label": "person",
        "confidence": 0.88,
        "bbox": [
          420,
          300,
          60,
          120
        ]
      }
    ]
  }
}


## Validation Rules

`send_metadata` is not a blind string concatenation. It parses and checks your `data_json` before it
builds the envelope, and raises when it cannot. This is a feature: a malformed payload fails at the
sender, where you can see it, rather than being silently discarded by a receiver you cannot debug.

| Rule | Failure |
| --- | --- |
| `type` must not be empty | `MetadataSender metadata type must not be empty` |
| `data_json` must be valid JSON | `MetadataSender data_json parse failed: ...` |
| `data_json` must be a JSON **object**, not an array or scalar | `MetadataSender data_json must be a JSON object` |

That last rule catches a common mistake. You have a list of detections, so it is tempting to pass the
list. Wrap it in an object with a named key instead: the receiver needs to know what the list *is*.

In [9]:
cases = [
    ("empty type", "", '{"objects": []}'),
    ("not valid json", "object-detection", "{objects: []}"),
    ("json array, not object", "object-detection", '[{"label": "car"}]'),
    ("json scalar, not object", "object-detection", '"car"'),
    ("valid", "object-detection", '{"objects": [{"label": "car", "bbox": [1, 2, 3, 4]}]}'),
]

for name, type_name, data_json in cases:
    try:
        result = sender.send_metadata(type_name, data_json, 0, "frame-0")
        print(f"{name:24s} -> sent: {result}")
    except Exception as exc:
        print(f"{name:24s} -> {type(exc).__name__}: {exc}")

empty type               -> RuntimeError: MetadataSender metadata type must not be empty
not valid json           -> RuntimeError: MetadataSender data_json parse failed: [json.exception.parse_error.101] parse error at line 1, column 2: syntax error while parsing object key - invalid literal; last read: '{o'; expected string literal
json array, not object   -> RuntimeError: MetadataSender data_json must be a JSON object
json scalar, not object  -> RuntimeError: MetadataSender data_json must be a JSON object
valid                    -> sent: True


## `send_raw_json` And When To Use It

`send_raw_json` sends your string exactly as given, with no envelope and no validation beyond what the
socket does. It is the right call in exactly two situations:

- You are replaying captured payloads and must not alter them.
- You are building the envelope yourself for performance, e.g. reusing a pre-serialized buffer at high
  frame rates.

For everything else, `send_metadata` is safer, and the validation it does is validation you would
otherwise have to write.

Note that with `send_raw_json` you are responsible for the required `type` and `data` fields. Omit them
and the receiver drops the message without telling you.

In [10]:
raw_payload = json.dumps({
    "type": "object-detection",
    "timestamp": int(time.time() * 1000),
    "frame_id": "frame-8",
    "data": {"objects": [{"id": "obj_1", "label": "truck", "confidence": 0.77, "bbox": [10, 10, 200, 150]}]},
})

print("sent:", sender.send_raw_json(raw_payload))
print("payload length:", len(raw_payload), "bytes")

sent: True
payload length: 185 bytes


## Payload Shapes The Insight Viewer Understands

The `type` string selects the overlay. Insight ships five renderers. Any other `type` still arrives at
the receiver, but nothing is drawn for it.

**Coordinates are in the pixel space of the encoded video frame**, so a `bbox` for a 1280x720 stream
uses values in `0..1280` and `0..720`. The viewer scales them to whatever size the tile happens to be.
This is why `FRAME_WIDTH`/`FRAME_HEIGHT` above matter: they are your coordinate system, and they must
be the resolution you gave `video_sender`.

`bbox` is `[x, y, width, height]`, with `x, y` at the **top-left corner**. It is not `[x1, y1, x2, y2]`.
Neat's `decode_bbox` returns corners, so a conversion is always required. This is the most common
overlay bug, and it shows up as boxes that are the right shape but land in the wrong place.

### `object-detection`

Boxes with labels. The workhorse.

```json
{
  "type": "object-detection",
  "timestamp": 1752230400000,
  "data": {
    "objects": [
      {"id": "obj_1", "label": "car", "confidence": 0.92, "bbox": [100, 100, 100, 80]}
    ]
  }
}
```

| Field | Required | Meaning |
| --- | --- | --- |
| `objects[].bbox` | yes | `[x, y, width, height]` in frame pixels. |
| `objects[].label` | yes | Text drawn above the box. |
| `objects[].confidence` | no | Shown next to the label. |
| `objects[].id` | no | Stable identity across frames. |

### `classification`

Whole-frame labels, drawn as a list in the corner.

```json
{
  "type": "classification",
  "data": {
    "top_classes": [
      {"label": "urban", "confidence": 0.91},
      {"label": "daytime", "confidence": 0.74}
    ]
  }
}
```

### `pose-estimation`

Keypoints and the skeleton between them.

```json
{
  "type": "pose-estimation",
  "data": {
    "poses": [
      {
        "id": "pose_1",
        "label": "person",
        "keypoints": [
          {"name": "nose", "x": 200, "y": 150, "confidence": 0.95},
          {"name": "left_shoulder", "x": 180, "y": 200, "confidence": 0.91}
        ]
      }
    ]
  }
}
```

### `segmentation`

Masks, as either a polygon or an RLE string.

```json
{
  "type": "segmentation",
  "data": {
    "segments": [
      {"id": "seg_1", "label": "road", "mask_format": "polygon",
       "mask": [[0, 500], [1280, 500], [1280, 720], [0, 720]]}
    ]
  }
}
```

| `mask_format` | `mask` value |
| --- | --- |
| `"polygon"` | A list of `[x, y]` points in frame pixels. |
| `"rle"` | A run-length-encoded mask string. |

### `tracking`

Boxes plus identity, so the viewer can draw motion trails.

```json
{
  "type": "tracking",
  "data": {
    "tracks": [
      {"id": "trk-1", "label": "car", "confidence": 0.9, "bbox": [300, 200, 80, 130]}
    ]
  }
}
```

The trail is built from the box centers of successive messages that share an `id`. A track whose `id`
changes every frame draws no trail, so `id` must be stable for tracking to mean anything.

## Builders For Each Type

Rather than hand-writing JSON at the call site, build the payload with a small function. It keeps the
schema in one place and makes the coordinate conversion explicit.

In [11]:
def object_detection_payload(boxes):
    """boxes: list of dicts with x1, y1, x2, y2, score, label."""
    objects = []
    for index, box in enumerate(boxes):
        x1, y1, x2, y2 = box["x1"], box["y1"], box["x2"], box["y2"]
        objects.append({
            "id": box.get("id", f"obj_{index + 1}"),
            "label": box["label"],
            "confidence": round(float(box["score"]), 3),
            # Corners -> top-left + size. The viewer expects [x, y, w, h].
            "bbox": [int(x1), int(y1), int(x2 - x1), int(y2 - y1)],
        })
    return {"objects": objects}


def classification_payload(top_classes):
    return {"top_classes": [
        {"label": label, "confidence": round(float(score), 3)} for label, score in top_classes
    ]}


def pose_payload(poses):
    """poses: list of lists of (name, x, y, confidence)."""
    return {"poses": [
        {
            "id": f"pose_{index + 1}",
            "label": "person",
            "keypoints": [
                {"name": name, "x": int(x), "y": int(y), "confidence": round(float(conf), 3)}
                for name, x, y, conf in keypoints
            ],
        }
        for index, keypoints in enumerate(poses)
    ]}


def segmentation_payload(segments):
    """segments: list of (label, polygon points)."""
    return {"segments": [
        {
            "id": f"seg_{index + 1}",
            "label": label,
            "mask_format": "polygon",
            "mask": [[int(x), int(y)] for x, y in polygon],
        }
        for index, (label, polygon) in enumerate(segments)
    ]}


def tracking_payload(tracks):
    """tracks: list of dicts with id, label, score, x1, y1, x2, y2."""
    return {"tracks": [
        {
            "id": track["id"],  # Must be stable across frames or no trail is drawn.
            "label": track["label"],
            "confidence": round(float(track["score"]), 3),
            "bbox": [int(track["x1"]), int(track["y1"]),
                     int(track["x2"] - track["x1"]), int(track["y2"] - track["y1"])],
        }
        for track in tracks
    ]}


sample_boxes = [
    {"x1": 100, "y1": 100, "x2": 260, "y2": 220, "score": 0.93, "label": "car"},
    {"x1": 700, "y1": 320, "x2": 780, "y2": 560, "score": 0.81, "label": "person"},
]
print(json.dumps(object_detection_payload(sample_boxes), indent=2))

{
  "objects": [
    {
      "id": "obj_1",
      "label": "car",
      "confidence": 0.93,
      "bbox": [
        100,
        100,
        160,
        120
      ]
    },
    {
      "id": "obj_2",
      "label": "person",
      "confidence": 0.81,
      "bbox": [
        700,
        320,
        80,
        240
      ]
    }
  ]
}


## Verify Locally With A UDP Receiver

Before pointing anything at Insight, prove the wire works. Bind a socket, send to it, and read back the
exact bytes. This has no dependency on Insight, the network, or a browser, so when it works you know the
sender and your payload builders are correct.

In [12]:
receiver = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
receiver.bind(("127.0.0.1", 0))
receiver.settimeout(2.0)
local_port = receiver.getsockname()[1]

loopback_opt = pyneat.MetadataSenderOptions()
loopback_opt.host = "127.0.0.1"
loopback_opt.channel = 3
loopback_opt.metadata_port_base = local_port - 3  # channel 3 => port base + 3 == local_port.

loopback = pyneat.MetadataSender(loopback_opt)
print("sending to", loopback.host(), loopback.metadata_port(), "| listening on", local_port)

loopback.send_metadata(
    "object-detection",
    json.dumps(object_detection_payload(sample_boxes)),
    int(time.time() * 1000),
    "frame-1",
)

datagram, _addr = receiver.recvfrom(65535)
parsed = json.loads(datagram.decode("utf-8"))

print()
print("received", len(datagram), "bytes")
print(json.dumps(parsed, indent=2))
receiver.close()

sending to 127.0.0.1 55921 | listening on 55921

received 241 bytes
{
  "data": {
    "objects": [
      {
        "bbox": [
          100,
          100,
          160,
          120
        ],
        "confidence": 0.93,
        "id": "obj_1",
        "label": "car"
      },
      {
        "bbox": [
          700,
          320,
          80,
          240
        ],
        "confidence": 0.81,
        "id": "obj_2",
        "label": "person"
      }
    ]
  },
  "frame_id": "frame-1",
  "timestamp": 1784186415834,
  "type": "object-detection"
}


Send one of each type through the same loopback and confirm every builder produces a valid envelope.

In [13]:
receiver = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
receiver.bind(("127.0.0.1", 0))
receiver.settimeout(2.0)
local_port = receiver.getsockname()[1]

loopback_opt = pyneat.MetadataSenderOptions()
loopback_opt.host = "127.0.0.1"
loopback_opt.channel = 0
loopback_opt.metadata_port_base = local_port
loopback = pyneat.MetadataSender(loopback_opt)

payloads = [
    ("object-detection", object_detection_payload(sample_boxes)),
    ("classification", classification_payload([("urban", 0.91), ("daytime", 0.74)])),
    ("pose-estimation", pose_payload([[("nose", 200, 150, 0.95), ("left_shoulder", 180, 200, 0.91)]])),
    ("segmentation", segmentation_payload([("road", [(0, 500), (1280, 500), (1280, 720), (0, 720)])])),
    ("tracking", tracking_payload([
        {"id": "trk-1", "label": "car", "score": 0.9, "x1": 300, "y1": 200, "x2": 380, "y2": 330},
    ])),
]

for type_name, data in payloads:
    loopback.send_metadata(type_name, json.dumps(data), int(time.time() * 1000), "frame-1")

for _ in payloads:
    datagram, _addr = receiver.recvfrom(65535)
    message = json.loads(datagram.decode("utf-8"))
    print(f"{message['type']:18s} {len(datagram):5d} bytes  data keys: {list(message['data'].keys())}")

receiver.close()

object-detection     241 bytes  data keys: ['objects']
classification       171 bytes  data keys: ['top_classes']
pose-estimation      248 bytes  data keys: ['poses']
segmentation         193 bytes  data keys: ['segments']
tracking             156 bytes  data keys: ['tracks']


## Datagram Size

One message is one datagram. UDP caps a datagram at 65507 bytes of payload, and paths in between often
fragment or drop well before that. A send that exceeds the limit raises rather than truncating.

In practice, only two payload types get big: segmentation with dense polygons, and detection with a
large `top_k`. If you are near the limit, reduce before you send. A viewer cannot usefully draw 500
boxes anyway.

In [14]:
def payload_size(type_name: str, data: dict) -> int:
    envelope = {"type": type_name, "timestamp": 0, "frame_id": "frame-0", "data": data}
    return len(json.dumps(envelope, separators=(",", ":")).encode("utf-8"))


for count in [1, 10, 100, 1000]:
    boxes = [
        {"x1": i, "y1": i, "x2": i + 50, "y2": i + 50, "score": 0.9, "label": "car"}
        for i in range(count)
    ]
    size = payload_size("object-detection", object_detection_payload(boxes))
    print(f"{count:5d} objects -> {size:7d} bytes  {'OK' if size < 65507 else 'TOO LARGE FOR ONE DATAGRAM'}")

    1 objects ->     148 bytes  OK
   10 objects ->     734 bytes  OK
  100 objects ->    6855 bytes  OK
 1000 objects ->   70756 bytes  TOO LARGE FOR ONE DATAGRAM


## Send To Insight And Confirm It Arrived

`push`/`send` returning `True` means the datagram left the socket. UDP will not tell you whether anyone
received it. Insight's ingest endpoint will.

Look at the `metadata` object for your channel:

| Counter | Meaning |
| --- | --- |
| `messages_received` | Insight's vf process received your datagram. If this is flat, the problem is host, port, or network. |
| `messages_forwarded` | vf pushed it to a browser over the WebRTC DataChannel. |

If `messages_received` grows but `messages_forwarded` stays flat, your sender is fine and no browser is
watching that channel. Open the viewer and the second counter starts moving. That distinction saves a
lot of pointless debugging.

In [16]:
def insight_get(path: str):
    ctx = ssl.create_default_context()
    ctx.check_hostname = False  # Insight uses a local mkcert certificate.
    ctx.verify_mode = ssl.CERT_NONE
    with urllib.request.urlopen(f"{INSIGHT_API}{path}", context=ctx, timeout=5) as response:
        return json.loads(response.read().decode("utf-8"))


live = pyneat.MetadataSender(make_metadata_options(CHANNEL))
print("sending 60 messages to", live.host(), live.metadata_port())

for index in range(60):
    x = 100 + (index * 12) % 900  # A box that sweeps left to right, so a frozen overlay is obvious.
    boxes = [{"x1": x, "y1": 300, "x2": x + 160, "y2": 460, "score": 0.9, "label": "demo"}]
    live.send_metadata(
        "object-detection",
        json.dumps(object_detection_payload(boxes)),
        int(time.time() * 1000),
        f"frame-{index}",
    )
    time.sleep(1 / 30)

try:
    stats = insight_get("/api/ingest/stats?all=1")
    for channel in stats.get("channels", []):
        if channel.get("channel") != CHANNEL:
            continue
        meta = channel.get("metadata", {})
        print()
        print("channel :", channel.get("channel"))
        print("messages_received :", meta.get("messages_received"))
        print("messages_forwarded:", meta.get("messages_forwarded"))
        if meta.get("messages_received") and not meta.get("messages_forwarded"):
            print("-> Metadata is arriving but no browser is attached. Open the Insight viewer.")
        break
    else:
        print("channel", CHANNEL, "not reported by Insight.")
except Exception as exc:
    print("Insight not reachable:", type(exc).__name__, exc)
    print("The loopback cells above still prove the sender works.")

sending 60 messages to 192.168.131.12 9100

channel : 0
messages_received : 123
messages_forwarded: 0
-> Metadata is arriving but no browser is attached. Open the Insight viewer.


## Pattern: Per-Frame Metadata In A Real Loop

In an application, you build the sender once and call it once per frame, next to the `video_run.push`
that sends the matching pixels. The `frame_id` is what ties the two together for anyone correlating
them later.

```python
video_opt = pyneat.VideoSenderOptions.h264_rtp_udp_from_raw(width, height, fps)
video_opt.host = INSIGHT_HOST
video_opt.channel = CHANNEL          # one channel variable ...

meta_opt = pyneat.MetadataSenderOptions()
meta_opt.host = INSIGHT_HOST
meta_opt.channel = CHANNEL           # ... used for both senders
metadata_sender = pyneat.MetadataSender(meta_opt)

for frame_index, frame in enumerate(frames):
    detections = detect(frame)
    video_run.push([make_nv12_tensor(frame_nv12, width, height)])
    metadata_sender.send_metadata(
        "object-detection",
        json.dumps(object_detection_payload(detections)),
        int(time.time() * 1000),
        f"frame-{frame_index}",
    )
```

Two habits worth forming:

- **Send metadata even when there is nothing to report.** An empty `{"objects": []}` clears the previous
  overlay. Skip the send and the viewer keeps drawing the last boxes it saw, which looks like the model
  is stuck on a ghost.
- **Send at most once per frame.** Metadata is cheap, but it is still a syscall on the frame path.

Notebook 07 puts this together with a real RTSP source and real detections.

In [17]:
empty_frame_payload = object_detection_payload([])
print("clearing payload:", json.dumps(empty_frame_payload))
print("size:", payload_size("object-detection", empty_frame_payload), "bytes")

live.send_metadata("object-detection", json.dumps(empty_frame_payload), int(time.time() * 1000), "frame-final")
print("overlay cleared")

clearing payload: {"objects": []}
size: 84 bytes
overlay cleared


## Custom Metadata Types

Nothing stops you from sending a `type` that Insight does not render. The datagram arrives, the counters
move, and the viewer simply draws nothing. That is useful when the real consumer is your own service,
a recorder, or a log.

The only hard requirements are the ones `send_metadata` enforces: a non-empty `type` and a `data` that
is a JSON object.

In [18]:
custom = {
    "zone_counts": {"entrance": 3, "checkout": 7},
    "alert": False,
    "dwell_seconds": {"trk-1": 12.4, "trk-2": 3.1},
}

print("sent:", live.send_metadata("zone-analytics", json.dumps(custom), int(time.time() * 1000), "frame-99"))
print("Insight will receive this but will not draw it; a custom consumer can read it.")

sent: True
Insight will receive this but will not draw it; a custom consumer can read it.


## Troubleshooting

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| `RuntimeError` at construction | Host does not resolve, or the socket cannot be opened. | Check `host`. Construction is where a bad address surfaces. |
| `data_json must be a JSON object` | You passed a list of detections. | Wrap it: `{"objects": [...]}`. |
| `data_json parse failed` | You passed a Python dict, not a JSON string. | `json.dumps(...)` it first. |
| `messages_received` stays at 0 | Wrong host, wrong port, or a firewall. | Run the loopback cell. If it passes, the sender is fine. |
| `messages_received` grows, `messages_forwarded` does not | No browser is attached to that channel. | Open the Insight viewer on the matching `src`. |
| Boxes drawn in the wrong place | Passed `[x1, y1, x2, y2]` instead of `[x, y, w, h]`. | Convert corners to top-left plus size. |
| Boxes are the wrong size, scaled consistently | Coordinates are in model space (640x640), not frame space. | Use `decode_bbox(..., clamp_to=(frame_w, frame_h))`. |
| Overlay lands on the wrong video tile | Video channel and metadata channel disagree. | Derive both from one `CHANNEL` variable. |
| Old boxes stay on screen | You stopped sending when there was nothing to report. | Send an empty `{"objects": []}` every frame. |
| Overlay lags the video | Sending metadata on a slower cadence than frames. | Send once per frame, right beside the video push. |

For a synthetic reference stream that is known to be correct, Insight ships a generator:

```bash
neat-insight-metadata-test --count 1 --types object-detection --fps 30
```

If the viewer draws its boxes but not yours, the difference is in your payload, not in Insight.

## What To Remember

- `MetadataSender` is a plain object, not a graph node. You call it from your loop.
- `metadata_port = metadata_port_base + channel`. Read it from `metadata_port()`, never compute it.
- Video channel and metadata channel must be the same number, or the overlay lands on the wrong tile.
- `send_metadata` builds the `type` / `timestamp` / `frame_id` / `data` envelope and validates `data`.
  `send_raw_json` does neither. Prefer `send_metadata`.
- `data_json` must be a JSON **object** string. Not a dict, not a list.
- `bbox` is `[x, y, width, height]` in **encoded-frame pixel space**. `decode_bbox` gives you corners,
  so a conversion is always needed.
- Insight renders `object-detection`, `classification`, `pose-estimation`, `segmentation`, and
  `tracking`. Any other type is delivered but not drawn.
- Send an empty payload when nothing is detected, or stale boxes stay on screen.
- UDP is fire-and-forget. A `True` return proves the datagram left, nothing more. Confirm with
  `/api/ingest/stats`.

## References

- Notebook 05, `05_video_sender_options.ipynb`, for the video half of the same stream.
- Notebook 07, `07_rtsp_decode_encode_metadata_to_insight.ipynb`, for both halves against a live camera.
- Core header: `include/nodes/io/MetadataSender.h`.
- Core docs: `docs/develop-apps/advanced-concepts/application-design/metadata_sender.md`.
- Insight reference generator: `neat-insight-metadata-test`.